# 8.7 通过时间反向传播

通过时间反向传播（BPTT）把 RNN 沿时间展开，再对所有时间步共享的参数求梯度。由于序列很长时计算图也很长，实践中常使用截断 BPTT。


In [ ]:
import torch


## 截断历史梯度


In [ ]:
def detach_state(state):
    """在相邻小批量之间截断梯度传播。"""
    if isinstance(state, torch.Tensor):
        return state.detach()
    return tuple(s.detach() for s in state)

state = (torch.randn(2, 4, requires_grad=True),)
new_state = detach_state(state)
print(new_state[0].requires_grad)


## 梯度爆炸和梯度消失

RNN 的梯度会沿时间步反复乘以隐藏状态相关的雅可比矩阵。很多次连乘后，梯度可能指数级变大或变小，因此训练 RNN 时常配合梯度裁剪。


In [ ]:
def clip_by_global_norm(grads, theta):
    norm = torch.sqrt(sum((g ** 2).sum() for g in grads))
    if norm > theta:
        grads = [g * theta / norm for g in grads]
    return grads

grads = [torch.randn(3, 3) * 10, torch.randn(2) * 10]
clipped = clip_by_global_norm(grads, theta=1.0)
print(torch.sqrt(sum((g ** 2).sum() for g in clipped)))


BPTT 的关键不是引入新的网络层，而是明确 RNN 参数在时间维度上共享，并用截断和梯度裁剪让训练可控。
